# Gold: Wikipedia Clickstream Top Pages (v1) — SQL

SQL-cell equivalent of `gold_wikipedia_clickstream_top_pages_v1.ipynb`,
using `%%sql` magic instead of the DataFrame API. Same result: reads the
`wikipedia.clickstream_enwiki` bronze table and persists
`wikipedia.gold_clickstream_top_pages_v1`.

Requires the bronze table to already exist — run
`data-generators/bronze_ingest_wikipedia_clickstream.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: top pages by incoming clicks

Aggregates by destination page (`curr`): total clicks (`sum(n)`) and count
of distinct referring pages (`prev`). `CREATE OR REPLACE TABLE` keeps this
re-runnable.

In [ ]:
%%sql
CREATE OR REPLACE TABLE wikipedia.gold_clickstream_top_pages_v1 AS
SELECT
    curr AS page,
    SUM(n) AS total_clicks,
    COUNT(DISTINCT prev) AS distinct_referrers
FROM wikipedia.clickstream_enwiki
GROUP BY curr
ORDER BY total_clicks DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_pages FROM wikipedia.gold_clickstream_top_pages_v1

In [ ]:
%%sql
SELECT * FROM wikipedia.gold_clickstream_top_pages_v1
ORDER BY total_clicks DESC
LIMIT 10